In [1]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
PAGEINDEX_API_KEY = "4785e7978988474bbd2f091b79c23797"


print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq key loaded:     ", "✅" if GROQ_API_KEY      else "❌ Missing!")

PageIndex key loaded: ✅
Groq key loaded:      ✅


CLIENT SETUP

In [3]:
from pageindex import PageIndexClient 
from groq import Groq 

pi_client   = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ PageIndex client ready")
print("✅ Groq client ready")

✅ PageIndex client ready
✅ Groq client ready


---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

**Why NO chunking?**  
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.


In [4]:
PDF_PATH = "Literature_review_dmt.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

📤 Uploading: Literature_review_dmt.pdf


In [5]:
print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

✅ Uploaded!
📋 Document ID: pi-cmppe0n2200si01p9ibbrj26o
   (Save this ID — you'll use it throughout the notebook)


In [6]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [7]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])


print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 7

🌲 Raw tree (first node):
{
  "title": "Abstract",
  "node_id": "0000",
  "page_index": 1,
  "summary": "This PRISMA 2020-compliant systematic review evaluates 32 deep learning studies (2024\u20132026) regarding grain and seed quality assessment. It analyzes diverse architectural families and crop types, highlighting high classification accuracy but significant deficiencies in dataset transparency, cross-dataset validation, and real-world deployment readiness. The review concludes by proposing seven priority research directions, including the adoption of standardized benchmarks and advanced hyperspectral modeling to improve future research impact.",
  "text": "# Abstract\n\nAutomated grain and seed quality assessment is essential for post-harvest food-chain integrity, yet prior systematic reviews have not systematically treated reproducibility, evaluation rigour, or deployment readiness as primary review dimensions in the 2024\u20132026 grain deep learning liter

TO FETCH ENTIRE HIGH LEVEL TREE

In [8]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Abstract  (p.1)
[0001] Contents  (p.3)
[0002] 2 Methodology  (p.6)
  └─ [0003] 2.1 Protocol and Registration  (p.6)
  └─ [0004] 2.2 Eligibility Criteria  (p.6)
  └─ [0005] 2.3 Information Sources and Search Strategies  (p.6)
  └─ [0006] 2.4 Study Selection Process  (p.7)
  └─ [0007] 2.5 Data Extraction  (p.8)
  └─ [0008] 2.6 Quality Assessment  (p.8)
  └─ [0009] 2.7 Limitations of the Review Methodology  (p.9)
[0010] 3 Results  (p.10)
  └─ [0011] 3.1 Study Selection  (p.10)
  └─ [0012] 3.2 Characteristics of Included Studies  (p.11)
  └─ [0013] 3.3 Grain Type Distribution  (p.13)
  └─ [0014] 3.4 Quality Assessment  (p.13)
  └─ [0015] 4.1 Maize and Corn Classification  (p.16)
    └─ [0016] 4.1.1 Custom and Architecture-Enhanced CNNs  (p.16)
    └─ [0017] 4.1.2 Attention and Transformer-Augmented Architectures  (p.17)
    └─ [0018] 4.1.3 Hyperspectral Deep Learning for Corn  (p.18)
    └─ [0019] 4.1.4 Object Detection and Transfer Learning  (p.19)
  └─ 

EDA

COUNT THE NODES

In [9]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 39
   Each node = one retrievable section of the document


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


In [10]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM via Groq.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree 
    structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

SEARCH OPERATION

In [11]:
query = "How was data extraction performed"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: How was data extraction performed

🧠 LLM Reasoning:
The query asks for the procedure used for data extraction. In the document tree, the section titled '2.5 Data Extraction' (node_id 0007) directly addresses this topic, describing the extraction schema and process. The parent '2 Methodology' section (node_id 0002) is a broader container but does not itself detail extraction methods. Therefore, the most relevant node is 0007, and optionally its parent 0002 for broader context.

🎯 Selected Node IDs: ['0007']


---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks


In [12]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

GENERATION METHOD

In [13]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

COMPLETE RAG PIPELINE

In [14]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [15]:
answer = vectorless_rag(
    query="What was the conclusion from the review",
    tree=pageindex_tree
)

🔍 Query: What was the conclusion from the review

🧠 Reasoning: The query asks for the conclusion of the review. The Conclusions section (node_id 0037) directly contains the concluding statements of the systematic review. The Abstract (node_id 0000) may also summa...
🎯 Retrieved node IDs: ['0037', '0000']
📄 Sections found: ['Abstract', '6 Conclusions']

📝 Answer:
The review concluded that, although deep‑learning models now achieve proof‑of‑concept accuracies above 90 % for grain and seed classification across all major crops and architectural families, the field’s progress is limited by a lack of shared, multi‑crop benchmarks, routine cross‑dataset evaluation, and embedded‑hardware validation; without these evaluation and data‑infrastructure improvements, performance claims cannot be reliably verified for real‑world, regulatory and commercial deployment. (Section ‘6 Conclusions’, Page 34)


In [16]:
answer = vectorless_rag(
    query="Summarize Maize and Corn Classification analysis",
    tree=pageindex_tree
)

🔍 Query: Summarize Maize and Corn Classification analysis

🧠 Reasoning: The query asks for a summary of the Maize and Corn Classification analysis. In the document tree, the section dedicated to this topic is node 0015 titled '4.1 Maize and Corn Classification'. Its detai...
🎯 Retrieved node IDs: ['0015', '0016', '0017', '0018', '0019']
📄 Sections found: ['4.1 Maize and Corn Classification', '4.1.1 Custom and Architecture-Enhanced CNNs', '4.1.2 Attention and Transformer-Augmented Architectures', '4.1.3 Hyperspectral Deep Learning for Corn', '4.1.4 Object Detection and Transfer Learning']

📝 Answer:
**Maize and corn classification – key take‑aways**

- **Dominant crop** – Maize/corn appears in 11 of the 32 reviewed studies, making it the most‑studied grain (4.1 Maize and Corn Classification, p. 16).  

- **Architectural breadth** – The crop has been tackled with a full spectrum of models: custom CNNs, attention‑augmented hybrids, hierarchical Vision Transformers, hyperspectral 3D‑CNNs a